# Classificação de Falhas em Linhas de Transmissão Elétrica

Este notebook apresenta um estudo de caso de mineração de dados para classificação supervisionada de falhas em linhas de transmissão elétrica.

## Descrição do Problema

O objetivo é identificar padrões em medições elétricas de corrente e tensão que permitam distinguir diferentes tipos de falha em linhas de transmissão. A tarefa é tratada como um problema de classificação supervisionada usando dados do Kaggle.

## Contexto do Dataset

Fonte do dataset: Kaggle, dataset `esathyaprakash/electrical-fault-detection-and-classification`.

O dataset contém variáveis elétricas como correntes e tensões de fase. No arquivo `classData.csv`, as colunas `G`, `C`, `B` e `A` indicam o envolvimento da falha e são combinadas em uma única variável-alvo multiclasse chamada `fault_type`.

## Importação das Bibliotecas

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## Configurações Globais

In [ ]:
KAGGLE_DATASET = "esathyaprakash/electrical-fault-detection-and-classification"
DATASET_DIRECTORY = None
DATASET_FILE_PATH = None
TARGET_COLUMN = "fault_type"
TEST_SIZE = 0.2
RANDOM_STATE = 42
MINIMUM_RECALL_PER_CLASS = 0.90
RESULTS_PATH = Path("model_comparison_results.csv")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## Download do Dataset

In [ ]:
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
DATASET_DIRECTORY = Path(dataset_path)

print("Caminho dos arquivos do dataset:", DATASET_DIRECTORY)
available_files = sorted(path for path in DATASET_DIRECTORY.rglob("*") if path.is_file())
available_files

In [ ]:
def select_dataset_file(dataset_directory):
    csv_files = sorted(dataset_directory.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("Nenhum arquivo CSV foi encontrado no diretório baixado do dataset.")

    preferred_names = ["classData.csv", "classdata.csv"]
    for preferred_name in preferred_names:
        for csv_file in csv_files:
            if csv_file.name == preferred_name:
                return csv_file

    return max(csv_files, key=lambda csv_file: csv_file.stat().st_size)


DATASET_FILE_PATH = select_dataset_file(DATASET_DIRECTORY)
print("Arquivo de dataset selecionado:", DATASET_FILE_PATH)

## Carregamento do Dataset

In [ ]:
raw_data = pd.read_csv(DATASET_FILE_PATH)

print("Linhas e colunas:", raw_data.shape)
print("Colunas:", raw_data.columns.tolist())
print("Coluna-alvo configurada:", TARGET_COLUMN)
display(raw_data.head())
display(raw_data.dtypes.to_frame("tipo"))

In [ ]:
def build_fault_type(row):
    label_columns = ["G", "C", "B", "A"]
    active_labels = [label for label in label_columns if int(row[label]) == 1]
    return "Sem falha" if not active_labels else "".join(active_labels)


def prepare_target(dataframe):
    label_columns = ["G", "C", "B", "A"]
    if TARGET_COLUMN in dataframe.columns:
        return dataframe.copy(), []
    if set(label_columns).issubset(dataframe.columns):
        prepared = dataframe.copy()
        prepared[TARGET_COLUMN] = prepared.apply(build_fault_type, axis=1)
        return prepared, label_columns

    candidate_columns = ["faultType", "Fault Type", "Fault_Type", "Output (S)", "Output"]
    for candidate_column in candidate_columns:
        if candidate_column in dataframe.columns:
            prepared = dataframe.rename(columns={candidate_column: TARGET_COLUMN}).copy()
            return prepared, []

    raise ValueError("A coluna-alvo não pôde ser identificada. Revise as colunas do dataset antes de continuar.")


data, target_source_columns = prepare_target(raw_data)
print("Colunas de origem da variável-alvo:", target_source_columns or [TARGET_COLUMN])
display(data[[TARGET_COLUMN]].head())

## Inspeção Inicial dos Dados

Esta seção verifica dimensões, tipos de dados, valores ausentes, linhas duplicadas, estatísticas descritivas e distribuição da variável-alvo.

In [ ]:
print("Dimensões do dataset:", data.shape)
print("Linhas duplicadas:", data.duplicated().sum())

display(data.info())
display(data.isna().sum().to_frame("valores_ausentes"))
display(data.describe(include="all").T)
display(data[TARGET_COLUMN].value_counts().to_frame("quantidade"))

## Análise Exploratória dos Dados

A distribuição das classes indica se a tarefa de classificação está balanceada. Histogramas e boxplots mostram como as variáveis de corrente e tensão variam, enquanto a matriz de correlação destaca relações entre os preditores numéricos.

In [ ]:
analysis_feature_data = data.drop(columns=[TARGET_COLUMN])
if target_source_columns:
    analysis_feature_data = analysis_feature_data.drop(columns=target_source_columns)

numeric_columns_for_analysis = analysis_feature_data.select_dtypes(include=np.number).columns.tolist()
class_counts = data[TARGET_COLUMN].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
class_counts.plot(kind="bar", ax=ax, color="#2f6f8f")
ax.set_title("Distribuição das classes")
ax.set_xlabel("Tipo de falha")
ax.set_ylabel("Registros")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

class_counts

In [ ]:
data[numeric_columns_for_analysis].hist(figsize=(12, 8), bins=30, color="#3b7a57")
plt.suptitle("Histogramas das variáveis numéricas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = data[numeric_columns_for_analysis].corr()

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Matriz de correlação")
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_yticklabels(correlation_matrix.columns)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(correlation_matrix)

In [ ]:
selected_boxplot_columns = numeric_columns_for_analysis[:6]
fig, axes = plt.subplots(len(selected_boxplot_columns), 1, figsize=(10, 3 * len(selected_boxplot_columns)))
if len(selected_boxplot_columns) == 1:
    axes = [axes]

for axis, column in zip(axes, selected_boxplot_columns):
    data.boxplot(column=column, by=TARGET_COLUMN, ax=axis, rot=45)
    axis.set_title(f"{column} por tipo de falha")
    axis.set_xlabel("Tipo de falha")
    axis.set_ylabel(column)

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
def summarize_outliers(dataframe, columns):
    summaries = []
    for column in columns:
        first_quartile = dataframe[column].quantile(0.25)
        third_quartile = dataframe[column].quantile(0.75)
        interquartile_range = third_quartile - first_quartile
        lower_bound = first_quartile - 1.5 * interquartile_range
        upper_bound = third_quartile + 1.5 * interquartile_range
        outlier_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
        summaries.append({"variavel": column, "outliers": outlier_count, "taxa_outliers": outlier_count / len(dataframe)})
    return pd.DataFrame(summaries).sort_values("taxa_outliers", ascending=False)


outlier_summary = summarize_outliers(data, numeric_columns_for_analysis)
display(outlier_summary)

print("Achado da análise exploratória: balanceamento das classes, dispersão por alvo, correlações e taxas de outliers devem ser avaliados em conjunto antes da interpretação dos modelos.")

## Pré-processamento

Linhas duplicadas são removidas antes da modelagem. Valores ausentes e transformações são tratados dentro de pipelines do scikit-learn para evitar vazamento de dados. Qualquer transformação que aprende a partir dos dados é ajustada somente nos dados de treino.

In [ ]:
modeling_data = data.drop_duplicates().copy()

feature_columns_to_drop = [TARGET_COLUMN] + target_source_columns
features = modeling_data.drop(columns=feature_columns_to_drop)
target = modeling_data[TARGET_COLUMN]

numeric_features = features.select_dtypes(include=np.number).columns.tolist()
categorical_features = features.select_dtypes(exclude=np.number).columns.tolist()


def build_preprocessor(training_features):
    numeric_training_features = training_features.select_dtypes(include=np.number).columns.tolist()
    categorical_training_features = training_features.select_dtypes(exclude=np.number).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, numeric_training_features),
        ("categorical", categorical_transformer, categorical_training_features)
    ])


print("Linhas após remoção de duplicatas:", len(modeling_data))
print("Colunas numéricas:", numeric_features)
print("Colunas categóricas:", categorical_features)

## Engenharia de Features

As novas variáveis usam relações físicas do sistema trifásico para representar componente de sequência zero, desequilíbrio entre fases e magnitude total de corrente e tensão.

In [ ]:
required_three_phase_columns = ["Ia", "Ib", "Ic", "Va", "Vb", "Vc"]
missing_three_phase_columns = [column for column in required_three_phase_columns if column not in features.columns]
if missing_three_phase_columns:
    raise ValueError(f"Colunas necessárias para engenharia de features não encontradas: {missing_three_phase_columns}")

# Classificação física das falhas: [0 0 0 0] = sem falha, [1 0 0 1] = LG (fase A e terra), [0 0 1 1] = LL (fases A e B), [1 0 1 1] = LLG (fases A, B e terra), [0 1 1 1] = LLL (três fases), [1 1 1 1] = LLLG (três fases e terra).
features = features.copy()
features["I_seq_zero"] = features[["Ia", "Ib", "Ic"]].mean(axis=1)
features["V_seq_zero"] = features[["Va", "Vb", "Vc"]].mean(axis=1)
features["I_unbalance"] = features[["Ia", "Ib", "Ic"]].std(axis=1)
features["V_unbalance"] = features[["Va", "Vb", "Vc"]].std(axis=1)
features["I_magnitude"] = np.sqrt((features[["Ia", "Ib", "Ic"]] ** 2).sum(axis=1))
features["V_magnitude"] = np.sqrt((features[["Va", "Vb", "Vc"]] ** 2).sum(axis=1))

numeric_features = features.select_dtypes(include=np.number).columns.tolist()
categorical_features = features.select_dtypes(exclude=np.number).columns.tolist()

engineered_features = [
    "I_seq_zero",
    "V_seq_zero",
    "I_unbalance",
    "V_unbalance",
    "I_magnitude",
    "V_magnitude",
]
print("Features criadas:", engineered_features)
print("Colunas numéricas atualizadas:", numeric_features)

## Separação em Treino e Teste

In [ ]:
minimum_class_count = target.value_counts().min()
stratify_target = target if minimum_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print("Linhas de treino:", X_train.shape[0])
print("Linhas de teste:", X_test.shape[0])

## Treinamento dos Modelos

In [ ]:
models = {
    "Árvore de Decisão": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Floresta Aleatória": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Regressão Logística": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
}

trained_models = {}
for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(X_train)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    trained_models[model_name] = pipeline
    print(f"Modelo treinado: {model_name}")

## Avaliação dos Modelos

In [ ]:
evaluation_rows = []
predictions = {}

report_column_names = {
    "precision": "precisão",
    "recall": "recall",
    "f1-score": "f1-score",
    "support": "suporte",
}
report_index_names = {
    "accuracy": "acurácia",
    "macro avg": "média macro",
    "weighted avg": "média ponderada",
}

for model_name, pipeline in trained_models.items():
    y_pred = pipeline.predict(X_test)
    predictions[model_name] = y_pred
    evaluation_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    })
    print("\n", model_name)
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    report_table = pd.DataFrame(report).T.rename(columns=report_column_names).rename(index=report_index_names)
    display(report_table)

results = pd.DataFrame(evaluation_rows).sort_values(["f1_score", "recall"], ascending=False).reset_index(drop=True)
display(results.rename(columns={
    "model": "modelo",
    "accuracy": "acurácia",
    "precision": "precisão",
    "recall": "recall",
    "f1_score": "f1-score",
}))

## Análise por Classe

Esta seção detalha o desempenho de cada modelo por classe de falha. Para cada modelo treinado, são exibidas a matriz de confusão e uma tabela com precisão, recall e F1-score por classe. Ao final da análise de cada modelo, são impressos alertas para classes com recall abaixo do limiar configurado em `MINIMUM_RECALL_PER_CLASS`.

In [ ]:
class_labels = sorted(y_test.unique())
per_class_reports = {}

for model_name, y_pred in predictions.items():
    print(f"\nAnálise por classe - {model_name}")

    matrix = confusion_matrix(y_test, y_pred, labels=class_labels)
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"Matriz de confusão - {model_name}")
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
    ax.set_xticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(class_labels)))
    ax.set_yticklabels(class_labels)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            ax.text(column_index, row_index, matrix[row_index, column_index], ha="center", va="center", color="black")

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    report = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    report_table = pd.DataFrame(report).T.loc[class_labels, ["precision", "recall", "f1-score", "support"]]
    report_table = report_table.rename(columns={
        "precision": "precisão",
        "recall": "recall",
        "f1-score": "f1-score",
        "support": "suporte",
    })
    per_class_reports[model_name] = report_table
    display(report_table)

    low_recall_rows = report_table[report_table["recall"] < MINIMUM_RECALL_PER_CLASS]
    for class_name, row in low_recall_rows.iterrows():
        print(
            f"ALERTA: {model_name} teve recall {row['recall']:.4f} "
            f"para a classe {class_name}, abaixo do limiar configurado de {MINIMUM_RECALL_PER_CLASS:.2f}."
        )

## Validação Cruzada

O F1-score ponderado é usado porque é adequado para classificação multiclasse e cenários com desbalanceamento entre classes.

In [ ]:
fold_count = min(5, int(y_train.value_counts().min()))
cross_validation_rows = []

if fold_count >= 2:
    cross_validator = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=RANDOM_STATE)
    for model_name, model in models.items():
        pipeline = Pipeline(steps=[("preprocessor", build_preprocessor(X_train)), ("model", model)])
        scores = cross_val_score(pipeline, X_train, y_train, cv=cross_validator, scoring="f1_weighted", n_jobs=1)
        cross_validation_rows.append({"model": model_name, "cv_f1_mean": scores.mean(), "cv_f1_std": scores.std()})

cross_validation_results = pd.DataFrame(cross_validation_rows)
display(cross_validation_results.rename(columns={
    "model": "modelo",
    "cv_f1_mean": "f1_médio_validação_cruzada",
    "cv_f1_std": "desvio_f1_validação_cruzada",
}))
if cross_validation_results.empty:
    print("A validação cruzada foi ignorada porque pelo menos uma classe tem menos de dois registros.")

## Comparação dos Modelos

In [ ]:
if not cross_validation_results.empty:
    results = results.merge(cross_validation_results, on="model", how="left")

results_for_report = results.rename(columns={
    "model": "modelo",
    "accuracy": "acurácia",
    "precision": "precisão",
    "recall": "recall",
    "f1_score": "f1-score",
    "cv_f1_mean": "f1_médio_validação_cruzada",
    "cv_f1_std": "desvio_f1_validação_cruzada",
})
results_for_report.to_csv(RESULTS_PATH, index=False)
best_model_name = results.iloc[0]["model"]
best_model = trained_models[best_model_name]

display(results_for_report)
print("Resultados salvos em:", RESULTS_PATH)
print("Melhor modelo:", best_model_name)

## Visualizações dos Resultados

In [ ]:
metric_columns = ["accuracy", "precision", "recall", "f1_score"]
results.set_index("model")[metric_columns].plot(kind="bar", figsize=(10, 5))
plt.title("Comparação de métricas entre modelos")
plt.ylabel("Pontuação")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# As matrizes de confusão de todos os modelos foram exibidas na seção Análise por Classe acima.
# A importância das variáveis é exibida na célula seguinte.


In [ ]:
def get_feature_names(fitted_pipeline):
    fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
    return fitted_preprocessor.get_feature_names_out()


def plot_feature_importance(model_name, top_n=12):
    pipeline = trained_models[model_name]
    model = pipeline.named_steps["model"]
    if not hasattr(model, "feature_importances_"):
        print(f"{model_name} não expõe feature_importances_.")
        return pd.DataFrame()

    importance_data = pd.DataFrame({
        "variavel": get_feature_names(pipeline),
        "importancia": model.feature_importances_,
    }).sort_values("importancia", ascending=False)

    plot_data = importance_data.head(top_n).sort_values("importancia")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(plot_data["variavel"], plot_data["importancia"], color="#8a5a44")
    ax.set_title(f"Importância das variáveis - {model_name}")
    ax.set_xlabel("Importância")
    plt.tight_layout()
    plt.show()
    return importance_data


feature_importance_by_model = {}
for model_name in ["Árvore de Decisão", "Floresta Aleatória", "Gradient Boosting"]:
    if model_name in trained_models:
        feature_importance_by_model[model_name] = plot_feature_importance(model_name)

## Interpretação Crítica

O melhor modelo é selecionado principalmente pelo F1-score ponderado, depois pelo recall e pelo comportamento da matriz de confusão. A acurácia é usada apenas como evidência complementar.

Para manter a execução adequada a uma entrega acadêmica em ambiente como Google Colab, este notebook usa somente o treinamento direto dos modelos de linha de base. A etapa pesada de otimização com `GridSearchCV` foi removida desta versão porque aumentava muito o tempo de execução e não era necessária para demonstrar corretamente o fluxo de mineração de dados, pré-processamento sem vazamento, avaliação e interpretação.

A Árvore de Decisão deve ser revisada com cuidado porque uma árvore única pode se ajustar demais aos padrões do treino. A Floresta Aleatória geralmente reduz variância ao combinar múltiplas árvores e tende a ser mais estável. O Gradient Boosting pode capturar padrões não lineares de forma sequencial.

O desbalanceamento entre classes pode distorcer a acurácia, por isso precisão, recall e F1-score ponderados são priorizados. As matrizes de confusão devem ser inspecionadas para identificar quais classes de falha são mais confundidas. A importância de variáveis dos modelos baseados em árvores ajuda a identificar quais medições de corrente e tensão mais contribuem para a classificação.

Limitações incluem o uso de um único dataset público, possíveis medições simuladas, engenharia de atributos limitada e ausência de um conjunto externo de validação. Melhorias futuras podem incluir validação com dados reais de operação, calibração da confiança dos modelos e otimização de hiperparâmetros em uma versão separada, caso haja maior orçamento computacional.

## Conclusão Final

Este estudo examinou a classificação de falhas em linhas de transmissão elétrica usando um dataset do Kaggle. O fluxo baixou e carregou os dados, inspecionou problemas de qualidade, explorou distribuição de classes e comportamento numérico, removeu duplicatas, criou pipelines de pré-processamento com prevenção de vazamento, treinou múltiplos classificadores, avaliou modelos, comparou resultados, visualizou matrizes de confusão e importância de variáveis e salvou a tabela comparativa em CSV.

O melhor modelo final deve ser interpretado por meio do F1-score ponderado, recall, erros da matriz de confusão e importância das variáveis. A mineração de dados é relevante nesse contexto porque ajuda a transformar medições elétricas em apoio prático ao diagnóstico de falhas para operação e manutenção de sistemas de potência.